In [ ]:
# =========================
# WAR IMPACT ON ECONOMY — FULL CLEANED SCRIPT
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set(font_scale=1.2)
plt.rcParams["figure.figsize"] = (10, 6)

# =========================
# 1. Load data
# =========================
df = pd.read_csv("war_economic_impact_dataset.csv")

# =========================
# 2. Inspect data
# =========================
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nHead:\n", df.head())
print("\nSample:\n", df.sample(5, random_state=42))
print("\nInfo:")
df.info()
print("\nDescribe (numeric):")
print(df.describe())
print("\nDescribe (all):")
print(df.describe(include="all"))
print("\nMissing per column:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# =========================
# 3–4. Clean, strip, standardize, replace NaN numeric with median
# =========================

# Clean column names
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(" ", "_")
              .str.replace(r"[^0-9a-zA-Z_]", "", regex=True)
)

# Strip string columns
str_cols = df.select_dtypes(include="object").columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip()

# Standardize text fields
for col in ["conflict_name", "conflict_type", "region", "status", "most_affected_sector", "primary_country"]:
    if col in df.columns:
        df[col] = df[col].str.title()

# Convert numeric candidates
num_candidates = [
    "pre_war_unemployment_",
    "during_war_unemployment_",
    "unemployment_spike_percentage_points",
    "youth_unemployment_change_",
    "pre_war_poverty_rate_",
    "during_war_poverty_rate_",
    "extreme_poverty_rate_",
    "food_insecurity_rate_",
    "households_fallen_into_poverty_estimate",
    "gdp_change_",
    "inflation_rate_",
    "currency_devaluation_",
    "cost_of_war_usd",
    "estimated_reconstruction_cost_usd",
    "informal_economy_size_pre_war_",
    "informal_economy_size_during_war_",
    "currency_black_market_rate_gap_",
    "black_market_activity_level"
]
for col in num_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Imputation (clean copy)
df_clean = df.copy()

# Numeric → median
num_cols = df_clean.select_dtypes(include=["float64","int64","Int64"]).columns
for col in num_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Categorical → mode / Unknown
cat_cols = df_clean.select_dtypes(include="object").columns
for col in cat_cols:
    mode = df_clean[col].mode()
    df_clean[col] = df_clean[col].fillna(mode.iloc[0] if not mode.empty else "Unknown")

print("\nMissing after imputation:\n", df_clean.isna().sum())

# =========================
# 5. Handle missing values & date parse
# =========================
for col in ["start_year", "end_year"]:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").astype("Int64")

# =========================
# PLOT 1 — GDP CHANGE BY CONFLICT
# =========================

if {"conflict_name","gdp_change_"}.issubset(df_clean.columns):

    plot_data = df_clean.sort_values("gdp_change_")

    plt.figure(figsize=(12,6))
    ax = sns.barplot(
        data=plot_data,
        x="conflict_name",
        y="gdp_change_",
        hue="conflict_name",
        palette="Set2",
        legend=False
    )

    plt.xticks(rotation=90)
    plt.title("GDP Change During War by Conflict")
    plt.xlabel("Conflict")
    plt.ylabel("GDP Change (%)")

    for p, val in zip(ax.patches, plot_data["gdp_change_"]):
        ax.annotate(
            f"{val:.1f}%",
            (p.get_x() + p.get_width()/2, p.get_height()),
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold"
        )

    avg_change = plot_data["gdp_change_"].mean()
    worst_conflict = plot_data.loc[plot_data["gdp_change_"].idxmin(), "conflict_name"]
    worst_value = plot_data["gdp_change_"].min()

    summary_text = (
        f"Summary:\n"
        f"Average GDP change: {avg_change:.1f}%\n"
        f"Worst impact: {worst_conflict} ({worst_value:.1f}%)\n"
        f"Most conflicts show GDP decline during war."
    )

    ax.annotate(
        summary_text,
        xy=(0.5, -0.25),
        xycoords="axes fraction",
        ha="center",
        va="top",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.9)
    )

    plt.tight_layout()
    plt.show()

    kpi1 = df_clean.loc[df_clean["gdp_change_"].idxmin(), ["conflict_name","gdp_change_"]]
    print("\nKPI 1 – Worst GDP change conflict:\n", kpi1)

# =========================
# PLOT 2 — UNEMPLOYMENT SPIKE
# =========================

required_cols = {"conflict_name","unemployment_spike_percentage_points"}
missing = required_cols - set(df_clean.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

plot_data = df_clean.sort_values("unemployment_spike_percentage_points", ascending=False)

plt.figure(figsize=(12,6))
ax = sns.barplot(
    data=plot_data,
    x="conflict_name",
    y="unemployment_spike_percentage_points",
    hue="conflict_name",
    palette="tab10",
    legend=False
)

plt.xticks(rotation=90)
plt.title("Unemployment Spike During War by Conflict")
plt.xlabel("Conflict")
plt.ylabel("Spike (percentage points)")

for p, val in zip(ax.patches, plot_data["unemployment_spike_percentage_points"]):
    ax.annotate(
        f"{val:.1f} pp",
        (p.get_x() + p.get_width()/2, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=8,
        fontweight="bold"
    )

avg_spike = plot_data["unemployment_spike_percentage_points"].mean()
max_conflict = plot_data.loc[plot_data["unemployment_spike_percentage_points"].idxmax(), "conflict_name"]
max_value = plot_data["unemployment_spike_percentage_points"].max()

summary_text = (
    f"Summary:\n"
    f"Avg unemployment spike: {avg_spike:.1f} pp\n"
    f"Highest spike: {max_conflict} ({max_value:.1f} pp)\n"
    f"Most conflicts show increased unemployment during war."
)

ax.annotate(
    summary_text,
    xy=(0.5, -0.25),
    xycoords="axes fraction",
    ha="center",
    va="top",
    fontsize=10,
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.9)
)

plt.tight_layout()
plt.show()

kpi2 = df_clean["unemployment_spike_percentage_points"].max()
print("\nKPI 2 – Max unemployment spike (pp):", kpi2)

# =========================
# PLOT 3 — POVERTY CHANGE
# =========================

if {"conflict_name","pre_war_poverty_rate_","during_war_poverty_rate_"}.issubset(df_clean.columns):

    df_clean["poverty_change_"] = df_clean["during_war_poverty_rate_"] - df_clean["pre_war_poverty_rate_"]
    plot_data = df_clean.sort_values("poverty_change_", ascending=False)

    plt.figure(figsize=(12,6))
    ax = sns.barplot(
        data=plot_data,
        x="conflict_name",
        y="poverty_change_",
        hue="conflict_name",
        palette="Set3",
        legend=False
    )

    plt.xticks(rotation=90)
    plt.title("Change in Poverty Rate Due to War by Conflict")
    plt.xlabel("Conflict")
    plt.ylabel("Poverty Change (percentage points)")

    for p, val in zip(ax.patches, plot_data["poverty_change_"]):
        ax.annotate(
            f"{val:.1f} pp",
            (p.get_x() + p.get_width()/2, p.get_height()),
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold"
        )

    avg_change = plot_data["poverty_change_"].mean()
    worst_conflict = plot_data.loc[plot_data["poverty_change_"].idxmax(), "conflict_name"]
    worst_value = plot_data["poverty_change_"].max()

    summary_text = (
        f"Summary:\n"
        f"Avg poverty change: {avg_change:.1f} pp\n"
        f"Largest increase: {worst_conflict} ({worst_value:.1f} pp)\n"
        f"Most conflicts show rising poverty during war."
    )

    ax.annotate(
        summary_text,
        xy=(0.5, -0.25),
        xycoords="axes fraction",
        ha="center",
        va="top",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.9)
    )

    plt.tight_layout()
    plt.show()

    kpi3 = df_clean["poverty_change_"].mean()
    print("\nKPI 3 – Average poverty rate change (pp):", kpi3)

    # ---------- Plot 4: Inflation rate by region (bar + KPI) ----------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

if {"region","inflation_rate_"}.issubset(df_clean.columns):

    # Compute average inflation per region
    infl_region = df_clean.groupby("region")["inflation_rate_"].mean().reset_index()

    # Sort for cleaner visualization
    infl_region = infl_region.sort_values("inflation_rate_", ascending=False)

    plt.figure(figsize=(12,6))
    ax = sns.barplot(
        data=infl_region,
        x="region",
        y="inflation_rate_",
        hue="region",
        palette="tab20",
        legend=False
    )

    plt.xticks(rotation=45, ha="right")
    plt.title("Average Inflation Rate During War by Region")
    plt.xlabel("Region")
    plt.ylabel("Inflation Rate (%)")

    # ---- Label each bar with increase/decrease ----
    for p, val in zip(ax.patches, infl_region["inflation_rate_"]):
        label = f"{val:.2f}%"
        ax.annotate(
            label,
            (p.get_x() + p.get_width()/2, p.get_height()),
            ha="center",
            va="bottom",
            fontsize=8,
            fontweight="bold"
        )

    # ---- Summary box (bottom center) ----
    avg_infl = infl_region["inflation_rate_"].mean()
    highest_region = infl_region.loc[infl_region["inflation_rate_"].idxmax(), "region"]
    highest_value = infl_region["inflation_rate_"].max()

    summary_text = (
        f"Summary:\n"
        f"Avg inflation rate: {avg_infl:.2f}%\n"
        f"Highest inflation: {highest_region} ({highest_value:.2f}%)\n"
        f"Most regions show elevated inflation during war."
    )

    ax.annotate(
        summary_text,
        xy=(0.5, -0.25),
        xycoords="axes fraction",
        ha="center",
        va="top",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.9)
    )

    plt.tight_layout()
    plt.show()

    # ---- KPI 4 ----
    kpi4 = infl_region.loc[infl_region["inflation_rate_"].idxmax()]
    print("\nKPI 4 – Region with highest inflation:\n", kpi4)

# ---------- Plot 5: Cost of war vs reconstruction cost (scatter) ----------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

if {"cost_of_war_usd","estimated_reconstruction_cost_usd","region"}.issubset(df_clean.columns):

    plt.figure(figsize=(12,6))
    ax = sns.scatterplot(
        data=df_clean,
        x="cost_of_war_usd",
        y="estimated_reconstruction_cost_usd",
        hue="region",
        palette="tab10",
        alpha=0.8
    )

    plt.title("Cost of War vs Estimated Reconstruction Cost")
    plt.xlabel("Cost of War (USD)")
    plt.ylabel("Reconstruction Cost (USD)")

    # ---- Label each point with % increase/decrease ----
    df_clean["pct_change"] = (
        (df_clean["estimated_reconstruction_cost_usd"] - df_clean["cost_of_war_usd"])
        / df_clean["cost_of_war_usd"]
    ) * 100

    for i, row in df_clean.iterrows():
        ax.annotate(
            f"{row['pct_change']:.1f}%",
            (row["cost_of_war_usd"], row["estimated_reconstruction_cost_usd"]),
            textcoords="offset points",
            xytext=(5,5),
            fontsize=8,
            fontweight="bold"
        )

    # ---- Summary box (top center) ----
    avg_pct_change = df_clean["pct_change"].mean()
    max_region = df_clean.loc[df_clean["pct_change"].idxmax(), "region"]
    max_value = df_clean["pct_change"].max()

    summary_text = (
        f"Summary:\n"
        f"Avg % change: {avg_pct_change:.1f}%\n"
        f"Largest increase: {max_region} ({max_value:.1f}%)\n"
        f"Reconstruction costs generally exceed war costs."
    )

    ax.annotate(
        summary_text,
        xy=(0.5, 1.05),
        xycoords="axes fraction",
        ha="center",
        va="bottom",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.9)
    )

    plt.tight_layout()
    plt.show()

    # ---- KPI 5 ----
    kpi5 = df_clean["cost_of_war_usd"].corr(df_clean["estimated_reconstruction_cost_usd"])
    print("\nKPI 5 – Corr(cost_of_war_usd, estimated_reconstruction_cost_usd):", kpi5)

# ---------- Plot 6: Informal economy pre vs during (by region, bar) ----------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

if {"region","informal_economy_size_pre_war_","informal_economy_size_during_war_"}.issubset(df_clean.columns):

    # Compute regional averages
    inf_region = df_clean.groupby("region")[[
        "informal_economy_size_pre_war_",
        "informal_economy_size_during_war_"
    ]].mean().reset_index()

    # Compute % increase/decrease
    inf_region["pct_change"] = (
        (inf_region["informal_economy_size_during_war_"] -
         inf_region["informal_economy_size_pre_war_"])
        / inf_region["informal_economy_size_pre_war_"]
    ) * 100

    # Plot
    plt.figure(figsize=(12,6))
    ax = inf_region.set_index("region")[[
        "informal_economy_size_pre_war_",
        "informal_economy_size_during_war_"
    ]].plot(
        kind="bar",
        color=["tab:green","tab:orange"]
    )

    plt.title("Informal Economy Size Pre- vs During War by Region")
    plt.xlabel("Region")
    plt.ylabel("Size (%) or index)")
    plt.xticks(rotation=45, ha="right")
    plt.legend(["Pre-war","During war"])

    # ---- Label each bar with % change ----
    for idx, row in inf_region.iterrows():
        ax.annotate(
            f"{row['pct_change']:.1f}%",
            (idx, row["informal_economy_size_during_war_"]),
            textcoords="offset points",
            xytext=(5,5),
            fontsize=8,
            fontweight="bold"
        )

    # ---- Summary box (top-left) ----
    avg_change = inf_region["pct_change"].mean()
    max_region = inf_region.loc[inf_region["pct_change"].idxmax(), "region"]
    max_value = inf_region["pct_change"].max()

    summary_text = (
        f"Summary:\n"
        f"Avg % change: {avg_change:.1f}%\n"
        f"Largest increase: {max_region} ({max_value:.1f}%)\n"
        f"Informal economy generally expands during war."
    )

    ax.annotate(
        summary_text,
        xy=(0.02, 1.02),
        xycoords="axes fraction",
        ha="left",
        va="bottom",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.9)
    )

    plt.tight_layout()
    plt.show()

    # ---- KPI 6 ----
    kpi6 = inf_region["pct_change"].mean()
    print("\nKPI 6 – Average informal economy increase:", kpi6)

# ---------- Plot 7: Correlation heatmap ----------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

corr_cols = [
    "gdp_change_",
    "unemployment_spike_percentage_points",
    "poverty_change_" if "poverty_change_" in df_clean.columns else None,
    "inflation_rate_",
    "currency_devaluation_",
    "food_insecurity_rate_",
    "black_market_activity_level"
]
corr_cols = [c for c in corr_cols if c is not None and c in df_clean.columns]

if len(corr_cols) >= 2:

    corr_matrix = df_clean[corr_cols].corr()

    plt.figure(figsize=(10,8))
    ax = sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap – Economic & Social War Impacts")
    plt.tight_layout()

    # ---- Compute mean values for summary ----
    pct_changes = {col: df_clean[col].mean() for col in corr_cols}

    summary_lines = ["Summary:"]
    for col, val in pct_changes.items():
        summary_lines.append(f"{col}: {val:.2f}")

    summary_text = "\n".join(summary_lines)

    # ---- Summary box (TOP FAR LEFT) ----
    ax.annotate(
        summary_text,
        xy=(-0.35, 1.15),          # farther left & above the heatmap
        xycoords="axes fraction",
        ha="left",
        va="top",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.9)
    )

    plt.show()

    # ---- KPI 7 ----
    corr_unstack = corr_matrix.abs().unstack()
    corr_unstack = corr_unstack[corr_unstack < 1]
    strongest = corr_unstack.sort_values(ascending=False).head(1)

    print("\nKPI 7 – Strongest correlation between indicators:\n", strongest)


# ---------- Plot 8: Line Chart Matrix ----------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

line_cols = [
    "gdp_change_",
    "unemployment_spike_percentage_points",
    "inflation_rate_",
    "currency_devaluation_",
    "food_insecurity_rate_"
]

if "poverty_change_" in df_clean.columns:
    line_cols.append("poverty_change_")

trend = df_clean.groupby("start_year")[line_cols].mean().reset_index()

plt.figure(figsize=(14,7))

for col in line_cols:
    plt.plot(
        trend["start_year"],
        trend[col],
        linewidth=2,
        label=col.replace("_", " ").title()
    )

plt.title("War vs Economic & Social Indicators Over Time")
plt.xlabel("Start Year")
plt.ylabel("Average Indicator Value")
plt.grid(True)
plt.legend(loc="upper left")

pct_changes = {}
for col in line_cols:
    first = trend[col].iloc[0]
    last = trend[col].iloc[-1]
    pct_changes[col] = ((last - first) / first) * 100 if first != 0 else np.nan

summary_lines = ["Summary:"]
for col, val in pct_changes.items():
    summary_lines.append(f"{col}: {val:.2f}% change")

summary_text = "\n".join(summary_lines)

plt.annotate(
    summary_text,
    xy=(0.5, -0.25),
    xycoords="axes fraction",
    ha="center",
    va="top",
    fontsize=10,
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.9)
)

plt.tight_layout()
plt.show()

corr_matrix = df_clean[line_cols].corr()
corr_unstack = corr_matrix.abs().unstack()
corr_unstack = corr_unstack[corr_unstack < 1]
strongest = corr_unstack.sort_values(ascending=False).head(1)

print("\nKPI 7 – Strongest correlation between indicators:\n", strongest)

# =========================
# EXECUTIVE SUMMARY
# =========================
exec_summary = """
EXECUTIVE SUMMARY – WAR ECONOMIC & SOCIAL IMPACT DATASET

Structure:
The dataset describes individual conflicts with fields for conflict name, type, region, start and end year,
status, primary country, and a rich set of economic and social indicators.

Economic impacts:
GDP change, unemployment spikes, and inflation rates show that most conflicts are associated with economic
deterioration.

Social impacts:
Poverty and food insecurity rates rise during war. Informal economy and black market activity increase.

Correlations:
The correlation heatmap shows meaningful links between GDP change, unemployment spikes, poverty changes,
inflation, and black market activity.

Limitations:
Missing numeric values were imputed using median; categorical values using mode.

Overall:
War consistently undermines economic stability, increases poverty and unemployment, expands informal and
black market activity, and often involves war profiteering.
"""

print(exec_summary)



ModuleNotFoundError: No module named 'pandas'